In [408]:
import seaborn as sns
import pandas as pd
import numpy as np
from pathlib import Path
from matplotlib import pyplot as plt
from matplotlib.ticker import MaxNLocator
import math
from pathlib import Path

from xarch_tokenizers.logging.report_utils import (
    load_predictions,
    load_all_samples,
    clean_model_name,
)
from xarch_tokenizers.logging.plot_utils import (
    setup_styles,
    get_new_6_fig,
    MODEL_TO_COLOR,
    get_new_fig,
)


setup_styles()
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [409]:
## same as exploratory plotting notebook `paper-benchmark-plots.ipynb`
def get_canonical_and_perturbed_df(
    samples,
    only_keep_canonical_true: bool = False,
    keep_remains_or_becomes_correct: bool = False,
    only_keep_perturbed_true: bool = False,
    verbose: bool = False,
):
    """returns filtered out samples too"""
    if (
        (only_keep_canonical_true and keep_remains_or_becomes_correct)
        or (only_keep_canonical_true and only_keep_perturbed_true)
        or (keep_remains_or_becomes_correct and only_keep_perturbed_true)
    ):
        raise ValueError(
            f"Please pass these exclusively, only_keep_canonical_true: true for canonical, model_name true, keep_remains_or_becomes_correct: True to include the ones where the perturbed version becomes correct"
        )
    if verbose:
        print(f"Processing {len(samples)} examples")
    canonical_samples = samples[samples["is_canonical"]]
    perturbed_samples = samples[~samples["is_canonical"]]
    canonical_values = canonical_samples[["set_id", "model_name", METRIC]].rename(
        columns={METRIC: f"canonical_{METRIC}"}
    )

    perturbed_samples = perturbed_samples.merge(
        canonical_values, on=["set_id", "model_name"], how="left"
    )
    perturbed_samples[f"canonical-perturbed_{METRIC}"] = (
        perturbed_samples[f"canonical_{METRIC}"] - perturbed_samples[METRIC]
    )

    if only_keep_canonical_true:
        correct_canonical_pairs = canonical_samples[canonical_samples[METRIC] == 1][
            ["set_id", "model_name"]
        ]
        if verbose:
            print(
                f"Cleaning samples, will be dropping entries (set_id-model pairs) where the canonical is predicted wrong."
            )
            print(
                f"Keeping {len(correct_canonical_pairs)} correct pairs, dropping {len(canonical_samples) - len(correct_canonical_pairs)} entries."
            )

        canonical_samples = canonical_samples[canonical_samples[METRIC] == 1]

        # Step 3: Filter perturbed_samples to keep only rows with correct canonical pairs
        # Method 1: Using merge (RECOMMENDED)
        perturbed_samples = perturbed_samples.merge(
            correct_canonical_pairs,
            on=["set_id", "model_name"],
            how="inner",  # Only keep rows that exist in both
        )
        samples = pd.concat((canonical_samples, perturbed_samples), ignore_index=True)

    if keep_remains_or_becomes_correct:
        # Get pairs where perturbed samples are correct (becomes or remains correct)
        correct_perturbed_pairs = perturbed_samples[perturbed_samples[METRIC] == 1][
            ["set_id", "model_name"]
        ].drop_duplicates()

        # Get pairs where canonical samples are correct (remains correct)
        correct_canonical_pairs = canonical_samples[canonical_samples[METRIC] == 1][
            ["set_id", "model_name"]
        ]

        # Combine: pairs that either remain correct OR become correct
        remains_or_becomes_correct = pd.concat(
            [correct_canonical_pairs, correct_perturbed_pairs]
        ).drop_duplicates()

        # Filter both datasets to only include these pairs
        canonical_samples = canonical_samples.merge(
            remains_or_becomes_correct, on=["set_id", "model_name"], how="inner"
        )

        perturbed_samples = perturbed_samples.merge(
            remains_or_becomes_correct, on=["set_id", "model_name"], how="inner"
        )
        samples = pd.concat((canonical_samples, perturbed_samples), ignore_index=True)
        if verbose:
            print(f"After filtering:{len(samples)}")
    if only_keep_perturbed_true:
        # Get pairs where perturbed samples are correct (becomes or remains correct)
        correct_perturbed_pairs = perturbed_samples[perturbed_samples[METRIC] == 1][
            ["set_id", "model_name"]
        ].drop_duplicates()

        # Get pairs where canonical samples are correct (remains correct)
        wrong_canonical_pairs = canonical_samples[canonical_samples[METRIC] == 0][
            ["set_id", "model_name"]
        ]
        # Find cases where canonical wrong AND perturbed correct (recovery cases)
        becomes_correct = pd.merge(
            wrong_canonical_pairs,
            correct_perturbed_pairs,
            how="inner",
            on=["set_id", "model_name"],
        )

        # Filter both datasets to only include recovery cases
        canonical_samples = canonical_samples.merge(
            becomes_correct, on=["set_id", "model_name"], how="inner"
        )

        perturbed_samples = perturbed_samples.merge(
            becomes_correct, on=["set_id", "model_name"], how="inner"
        )

        samples = pd.concat([canonical_samples, perturbed_samples], ignore_index=True)
        if verbose:
            print(f"After filtering: {len(samples)}")

    return samples, canonical_samples, perturbed_samples

# Let's Create Summary

In [410]:
OUTPUT_DIR = Path("./output/results-v5")
OUTPUT_DIR_ = OUTPUT_DIR / "summary"
METRIC = "acc_norm"
# read all samples
base_dir = Path("../results/paper-v5").resolve().absolute()

# # keeps set_id, model pairs that are correct for canonical
# suffix = "(Still correct)"
# only_keep_canonical_true = True
# keep_remains_or_becomes_correct = False
# only_keep_perturbed_true = False

# suffix = "(Remains or Becomes Correct)"
# only_keep_canonical_true = False
# keep_remains_or_becomes_correct = True
# only_keep_perturbed_true = False

# suffix = "(Flipped to correct)"
# only_keep_canonical_true = False
# keep_remains_or_becomes_correct = False
# only_keep_perturbed_true = True

# suffix = "(All)"
# only_keep_canonical_true = False
# keep_remains_or_becomes_correct = False
# only_keep_perturbed_true = False

code_pattern = "*"
title_pattern = "All Datasets"
OUTPUT_DIR = OUTPUT_DIR_ / "all"

OUTPUT_DIR = OUTPUT_DIR
OUTPUT_DIR.mkdir(exist_ok=True, parents=True)
base_dir.exists()
pred_files = list([p for p in base_dir.rglob(f"samples{code_pattern}.jsonl")])
len(pred_files), pred_files[:5]


(2528,
 [PosixPath('/Users/gsaltintas/code/phd/tokenizers/results/paper-v5/r-three__supertoken_models-llama_facebook-xglm-564M/samples_tokenizer_robustness_completion_general_currency_symbol_2025-09-20T10-09-51.814046.jsonl'),
  PosixPath('/Users/gsaltintas/code/phd/tokenizers/results/paper-v5/r-three__supertoken_models-llama_facebook-xglm-564M/samples_tokenizer_robustness_completion_farsi_canonical_2025-09-20T16-14-33.757080.jsonl'),
  PosixPath('/Users/gsaltintas/code/phd/tokenizers/results/paper-v5/r-three__supertoken_models-llama_facebook-xglm-564M/samples_tokenizer_robustness_completion_chinese_ocr_errors_2025-09-20T11-23-04.491388.jsonl'),
  PosixPath('/Users/gsaltintas/code/phd/tokenizers/results/paper-v5/r-three__supertoken_models-llama_facebook-xglm-564M/samples_tokenizer_robustness_completion_italian_typographical_errors_2025-09-20T10-31-24.441216.jsonl'),
  PosixPath('/Users/gsaltintas/code/phd/tokenizers/results/paper-v5/r-three__supertoken_models-llama_facebook-xglm-564M/s

In [411]:
## Read all samples into a DataFrame
## exclude general dataset for now
all_samples = load_all_samples(
    base_dir,
    patterns=[code_pattern],
    exclude_patterns=["general"],
    flatten_doc=True,
    match_date=False,
    simplify_df=True,
)

2366 [PosixPath('/Users/gsaltintas/code/phd/tokenizers/results/paper-v5/r-three__supertoken_models-llama_facebook-xglm-564M/samples_tokenizer_robustness_completion_farsi_canonical_2025-09-20T16-14-33.757080.jsonl'), PosixPath('/Users/gsaltintas/code/phd/tokenizers/results/paper-v5/r-three__supertoken_models-llama_facebook-xglm-564M/samples_tokenizer_robustness_completion_chinese_ocr_errors_2025-09-20T11-23-04.491388.jsonl'), PosixPath('/Users/gsaltintas/code/phd/tokenizers/results/paper-v5/r-three__supertoken_models-llama_facebook-xglm-564M/samples_tokenizer_robustness_completion_italian_typographical_errors_2025-09-20T10-31-24.441216.jsonl'), PosixPath('/Users/gsaltintas/code/phd/tokenizers/results/paper-v5/r-three__supertoken_models-llama_facebook-xglm-564M/samples_tokenizer_robustness_completion_italian_grammatical_errors_2025-09-20T10-31-24.441216.jsonl'), PosixPath('/Users/gsaltintas/code/phd/tokenizers/results/paper-v5/r-three__supertoken_models-llama_facebook-xglm-564M/samples_t

We want a summary table like below

| filtering_mode   | model_name   | task                                                      | task_pretty_name     |   canonical_count |   num_samples | category                 | langs    |   acc_norm |   acc_norm_std |      acc |   acc_std | canonical_task_name                            |   canonical_acc |   canonical_acc_norm |
|:-----------------|:-------------|:----------------------------------------------------------|:---------------------|------------------:|--------------:|:-------------------------|:---------|-----------:|---------------:|---------:|----------:|:-----------------------------------------------|----------------:|---------------------:|
| no_filter        | Aya          | tokenizer_robustness_completion_stem_fullwidth_characters | Fullwidth Characters |                17 |            17 | Structural Text Elements | eng_Latn |   0.294118 |       0.469668 | 0.352941 |  0.492592 | tokenizer_robustness_completion_stem_canonical |        0.882353 |             0.823529 |
| no_filter        | BLOOM        | tokenizer_robustness_completion_stem_fullwidth_characters | Fullwidth Characters |                17 |            17 | Structural Text Elements | eng_Latn |   0.411765 |       0.5073   | 0.294118 |  0.469668 | tokenizer_robustness_completion_stem_canonical |        0.882353 |             0.823529 |
| no_filter        | ByT5         | tokenizer_robustness_completion_stem_fullwidth_characters | Fullwidth Characters |                17 |            17 | Structural Text Elements | eng_Latn |   0.411765 |       0.5073   | 0.294118 |  0.469668 | tokenizer_robustness_completion_stem_canonical |        0.882353 |             0.823529 |
| no_filter        | Comma        | tokenizer_robustness_completion_stem_fullwidth_characters | Fullwidth Characters |                17 |            17 | Structural Text Elements | eng_Latn |   0.352941 |       0.492592 | 0.294118 |  0.469668 | tokenizer_robustness_completion_stem_canonical |        0.882353 |             0.823529 |
| no_filter        | GPT-2        | tokenizer_robustness_completion_stem_fullwidth_characters | Fullwidth Characters |                17 |            17 | Structural Text Elements | eng_Latn |   0.352941 |       0.492592 | 0.235294 |  0.437237 | tokenizer_robustness_completion_stem_canonical |        0.941176 |             0.823529 |
| no_filter        | GPT-4o       | tokenizer_robustness_completion_stem_fullwidth_characters | Fullwidth Characters |                17 |            17 | Structural Text Elements | eng_Latn |   0.352941 |       0.492592 | 0.352941 |  0.492592 | tokenizer_robustness_completion_stem_canonical |        1        |             0.941176 |
| no_filter        | Gemma-2      | tokenizer_robustness_completion_stem_fullwidth_characters | Fullwidth Characters |                17 |            17 | Structural Text Elements | eng_Latn |   0.411765 |       0.5073   | 0.294118 |  0.469668 | tokenizer_robustness_completion_stem_canonical |        0.941176 |             0.882353 |
| no_filter        | Llama-3.2    | tokenizer_robustness_completion_stem_fullwidth_characters | Fullwidth Characters |                17 |            17 | Structural Text Elements | eng_Latn |   0.352941 |       0.492592 | 0.294118 |  0.469668 | tokenizer_robustness_completion_stem_canonical |        0.882353 |             0.882353 |
| no_filter        | Phi-3        | tokenizer_robustness_completion_stem_fullwidth_characters | Fullwidth Characters |                17 |            17 | Structural Text Elements | eng_Latn |   0.411765 |       0.5073   | 0.235294 |  0.437237 | tokenizer_robustness_completion_stem_canonical |        0.882353 |             0.823529 |
| no_filter        | Qwen-3       | tokenizer_robustness_completion_stem_fullwidth_characters | Fullwidth Characters |                17 |            17 | Structural Text Elements | eng_Latn |   0.294118 |       0.469668 | 0.294118 |  0.469668 | tokenizer_robustness_completion_stem_canonical |        0.882353 |             0.823529 |
| no_filter        | Tekken       | tokenizer_robustness_completion_stem_fullwidth_characters | Fullwidth Characters |                17 |            17 | Structural Text Elements | eng_Latn |   0.235294 |       0.437237 | 0.176471 |  0.392953 | tokenizer_robustness_completion_stem_canonical |        0.941176 |             0.882353 |
| no_filter        | TokenMonster | tokenizer_robustness_completion_stem_fullwidth_characters | Fullwidth Characters |                17 |            17 | Structural Text Elements | eng_Latn |   0.294118 |       0.469668 | 0.294118 |  0.469668 | tokenizer_robustness_completion_stem_canonical |        0.941176 |             0.764706 |
| no_filter        | XGLM         | tokenizer_robustness_completion_stem_fullwidth_characters | Fullwidth Characters |                17 |            17 | Structural Text Elements | eng_Latn |   0.823529 |       0.392953 | 0.882353 |  0.332106 | tokenizer_robustness_completion_stem_canonical |        0.882353 |             0.823529 |
| no_filter        | mBERT        | tokenizer_robustness_completion_stem_fullwidth_characters | Fullwidth Characters |                17 |            17 | Structural Text Elements | eng_Latn |   0.235294 |       0.437237 | 0.294118 |  0.469668 | tokenizer_robustness_completion_stem_canonical |        0.882353 |             0.764706 |


started with this

|          |                     |            |     |          |                   |                     |             |                  |                     |                     |
| -------- | ------------------- | ---------- | --- | -------- | ----------------- | ------------------- | ----------- | ---------------- | ------------------- | ------------------- |
| Category | Perturbation (Task) | Model name | acc | acc_norm | number of samples | filter              | acc_std_err | acc_norm_std_err | number of canonical | number of perturbed |
|          | romanization        | Comma      |     |          |                   | only_canonical_true |             |                  |                     |                     |
|          | dialect             | Comma      |     |          |                   | only_canonical_true |             |                  |                     |                     |
|          | romanization        | Comma      |     |          |                   | no_filter           |             |                  |                     |                     |
|          | dialect             | Comma      |     |          |                   | no_filter           |             |                  |                     |                     |
|          |                     |            |     |          |                   | only_canonical      |             |                  |                     |                     |
|          |                     |            |     |          |                   | only_perturbed      |             |                  |                     |                     |								

In [412]:
import warnings

from category_mapping import SUBCATEGORY_TO_CATEGORY
from typing import List

FILTERS = [
    "no_filter",
    "only_canonical_correct",
    "canonical_wrong_at_least_one_perturb_correct",
    "remains_or_becomes_correct",
]
keys = {
    "no_filter": (False, False, False),
    "only_canonical_correct": (True, False, False),
    "canonical_wrong_at_least_one_perturb_correct": (False, False, True),
    "remains_or_becomes_correct": (False, True, False),
}


def get_group_name(task_name):
    if "stem" in task_name:
        return "tokenizer_robustness_completion_stem"
    elif "english" in task_name:
        return "tokenizer_robustness_completion_english"
    elif "turkish" in task_name:
        return "tokenizer_robustness_completion_turkish"
    elif "farsi" in task_name:
        return "tokenizer_robustness_completion_farsi"
    elif "italian" in task_name:
        return "tokenizer_robustness_completion_italian"
    elif "chinese" in task_name:
        return "tokenizer_robustness_completion_chinese"
    elif "math" in task_name:
        return "tokenizer_robustness_completion_math"
    elif "general" in task_name:
        return "tokenizer_robustness_completion_general"


all_tasks = all_samples["task_pretty_name"].unique()
all_tasks = all_samples["task"].unique()
model_names = all_samples["model_name"].unique()
## cleanup secondary categories
all_samples["final_category"] = all_samples["category"].apply(
    lambda x: str(x).split(",")[0]
)
### TODO: replace later
# ## stem doesn't have lang for some reason, hacking it for now
# all_samples["lang"]=all_samples.apply(lambda row: "eng_Latn" if row["lang"] )


def get_summaries(subcategories: List[str] = None):
    all_summaries = []
    for filtering_mode in FILTERS:
        #################### filtering & clean-up ####################
        # don't rely on canonical_df, perturbed_df, always use samples
        samples, canonical_df, perturbed_df = get_canonical_and_perturbed_df(
            all_samples, *keys[filtering_mode], verbose=False
        )
        ## remove duplicate results just in case they were run twice
        dup_keys = [
            "model_name",
            "task",
            "subcategories",
            "lang",
            "set_id",
            "var_id",
            "question",
        ]
        duplicate_count = (
            samples.groupby(dup_keys, as_index=False).size()["size"] > 1
        ).sum()
        print(f"N duplicates: {duplicate_count}")
        samples = samples.drop_duplicates(subset=dup_keys)
        metadata = {"filtering_mode": filtering_mode}
        #################### process results for each model ####################
        for model in model_names:
            df_filter_ = samples["model_name"] == model
            metadata["model_name"] = model
            model_df = samples[df_filter_]
            #################### process results for each model & task pair ####################
            for task in all_tasks:
                tmp = model_df[model_df["task"] == task]
                if len(tmp) == 0 and filtering_mode == "no_filter":
                    warnings.warn(
                        f"Oh noo, check for model: {model}, task: {task} if it is run, the df is empty, skipping for now..."
                    )
                if len(tmp) == 0:
                    continue
                task_pretty_name = tmp["task_pretty_name"].unique()[0]
                if subcategories and task_pretty_name not in subcategories:
                    continue
                #################### extract canonical information ####################
                task_group_name = get_group_name(task)
                cor_canonical_task_name = f"{task_group_name}_canonical"
                ## apply filtering: canonical examples, match set ids and match tasks group name
                corr_canonical = model_df[
                    model_df["is_canonical"]
                    & model_df["set_id"].isin(tmp["set_id"].unique())
                    # & model_df[model_df["task"].str.startswith(task_group_name)]
                ]
                canonical_count = len(corr_canonical)

                if len(tmp) == 0:
                    pass
                    print(f"Empty df for task {task} under filter: {filtering_mode}")
                    continue
                category = tmp["final_category"].unique()
                category = SUBCATEGORY_TO_CATEGORY.get(task_pretty_name, "")
                metadata.update(
                    {
                        "task": task,
                        "task_pretty_name": task_pretty_name,
                        "canonical_count": canonical_count,
                        "num_samples": len(tmp),
                        "category": category,
                        "langs": ",".join([l for l in tmp["lang"].unique() if l != ""]),
                    }
                )
                # compute accuracy metrics
                acc_norm = tmp["acc_norm"].mean()
                acc_norm_std = tmp["acc_norm"].std()
                acc = tmp["acc"].mean()
                acc_std = tmp["acc"].std()
                summary = {
                    "acc_norm": tmp["acc_norm"].mean(),
                    "acc_norm_std": tmp["acc_norm"].std(),
                    "acc": tmp["acc"].mean(),
                    "acc_std": tmp["acc"].std(),
                    "canonical_task_name": cor_canonical_task_name,
                    "canonical_acc": corr_canonical["acc"].mean(),
                    "canonical_acc_norm": corr_canonical["acc_norm"].mean(),
                    "group_name": task_group_name,
                }
                all_summaries.append(metadata | summary)

    all_summaries = pd.DataFrame.from_records(all_summaries)
    return all_summaries


all_summaries = get_summaries()

N duplicates: 30604
N duplicates: 61193
N duplicates: 21964
N duplicates: 63314


In [413]:
# langs = all_summaries["langs"].unique()
# langs
corr_canonical


,doc_id,target,metrics,acc,acc_norm,model_name,tokenizer_name,task,question,choices,...,token_counts.tokenmonster/englishcode-32000-consistent-v1,question_general_category,categories,task_pretty_name,vocab_bucket,is_canonical,var_id,final_category,canonical_acc_norm,canonical-perturbed_acc_norm
4428,8,0,"[acc, acc_norm]",0,1,mBERT,google-bert/bert-base-multilingual-cased,tokenizer_robustness_completion_stem_canonical,"int(0, 1) represents","[definite integral from 0 to 1, indefinite int...",...,6.0,Mathematics,NaN,Canonical,Medium,True,0.0,nan,NaN,NaN
4430,9,1,"[acc, acc_norm]",1,1,mBERT,google-bert/bert-base-multilingual-cased,tokenizer_robustness_completion_stem_canonical,A polygon with three sides is called a,"[square, triangle, circle, rectangle]",...,6.0,Mathematics/Geometry,NaN,Canonical,Medium,True,0.0,nan,NaN,NaN
4448,18,3,"[acc, acc_norm]",1,1,mBERT,google-bert/bert-base-multilingual-cased,tokenizer_robustness_completion_stem_canonical,The chemical formula H2SO4 represents,"[hydrogen sulfate, sulfur tetroxide, bisulfate...",...,8.0,Chemistry,NaN,Canonical,Medium,True,0.0,nan,NaN,NaN
4450,19,0,"[acc, acc_norm]",1,1,mBERT,google-bert/bert-base-multilingual-cased,tokenizer_robustness_completion_stem_canonical,The chemical formula CH4 represents,"[methane, water, benzene, oxygen]",...,7.0,Chemistry,NaN,Canonical,Medium,True,0.0,nan,NaN,NaN
4452,20,0,"[acc, acc_norm]",0,1,mBERT,google-bert/bert-base-multilingual-cased,tokenizer_robustness_completion_stem_canonical,The chemical formula CO2 represents,"[carbon dioxide, carbon monoxide, oxygen, carbon]",...,6.0,Chemistry,NaN,Canonical,Medium,True,0.0,nan,NaN,NaN
4454,21,1,"[acc, acc_norm]",1,1,mBERT,google-bert/bert-base-multilingual-cased,tokenizer_robustness_completion_stem_canonical,The chemical formula O2 represents,"[water, oxygen, ozone, air]",...,6.0,Chemistry,NaN,Canonical,Medium,True,0.0,nan,NaN,NaN
4456,22,0,"[acc, acc_norm]",1,1,mBERT,google-bert/bert-base-multilingual-cased,tokenizer_robustness_completion_stem_canonical,The chemical formula H2 represents,"[hydrogen, water, helium, gas]",...,6.0,Chemistry,NaN,Canonical,Medium,True,0.0,nan,NaN,NaN
4458,23,2,"[acc, acc_norm]",1,1,mBERT,google-bert/bert-base-multilingual-cased,tokenizer_robustness_completion_stem_canonical,The chemical formula N2 represents,"[ammonia, nitrate, nitrogen, air]",...,6.0,Chemistry,NaN,Canonical,Medium,True,0.0,nan,NaN,NaN
4460,24,1,"[acc, acc_norm]",1,1,mBERT,google-bert/bert-base-multilingual-cased,tokenizer_robustness_completion_stem_canonical,The chemical formula NH3 represents,"[nitrogen, ammonia, hydrogen, gas]",...,7.0,Chemistry,NaN,Canonical,Medium,True,0.0,nan,NaN,NaN
4464,26,0,"[acc, acc_norm]",1,1,mBERT,google-bert/bert-base-multilingual-cased,tokenizer_robustness_completion_stem_canonical,The chemical formula CaCO3 represents,"[calcium carbonate, calcium oxide, carbon diox...",...,8.0,Chemistry,NaN,Canonical,Medium,True,0.0,nan,NaN,NaN


In [414]:
## sanity check
print(
    all_summaries[
        all_summaries["task_pretty_name"] == "Fullwidth Characters"
    ].to_markdown(index=False)
)


| filtering_mode                               | model_name   | task                                                      | task_pretty_name     |   canonical_count |   num_samples | category                 | langs    |   acc_norm |   acc_norm_std |      acc |    acc_std | canonical_task_name                            |   canonical_acc |   canonical_acc_norm | group_name                           |
|:---------------------------------------------|:-------------|:----------------------------------------------------------|:---------------------|------------------:|--------------:|:-------------------------|:---------|-----------:|---------------:|---------:|-----------:|:-----------------------------------------------|----------------:|---------------------:|:-------------------------------------|
| no_filter                                    | Aya          | tokenizer_robustness_completion_stem_fullwidth_characters | Fullwidth Characters |                17 |            17 | Structura

In [415]:
save_path = OUTPUT_DIR / "summary.tsv"

all_summaries.to_csv(save_path, sep="\t")
print(f"File saved at \n{save_path}")

File saved at 
output/results-v5/summary/all/summary.tsv


# Pivot Views

In [416]:
import pandas as pd
import numpy as np


# Modified function to exclude aggregation rows
def highlight_extremes(s):
    """
    Highlight the max value in green and min value in red for each column
    Excludes aggregation rows from min/max calculation
    """
    if s.dtype != "object":  # Only apply to numeric columns
        # Define patterns that identify aggregation rows
        aggregation_patterns = [
            "MEAN",
            "STD",
            "MIN",
            "MAX",
            "MEDIAN",
            "RANGE",
            "TOP_3_AVG",
            "BOTTOM_3_AVG",
            "TASK_WEIGHTED_AVG",
            "BEST_MODEL_PER_TASK",
            "WORST_MODEL_PER_TASK",
            "─",
            "═",
            "___",  # Visual separators
        ]

        # Filter out aggregation rows for min/max calculation
        model_only_series = s.copy()
        model_indices_to_exclude = []

        for idx in s.index:
            # Check if index matches any aggregation pattern
            if any(pattern in str(idx).upper() for pattern in aggregation_patterns):
                model_indices_to_exclude.append(idx)

        # Remove aggregation rows from calculation
        model_only_series = model_only_series.drop(
            model_indices_to_exclude, errors="ignore"
        )

        # Calculate min/max only from model rows
        if len(model_only_series) > 0:
            max_val = model_only_series.max()
            min_val = model_only_series.min()
        else:
            max_val = None
            min_val = None

        styles = []
        for idx, val in s.items():
            # Don't highlight aggregation rows at all
            if any(pattern in str(idx).upper() for pattern in aggregation_patterns):
                styles.append("")
            elif pd.isna(val):
                styles.append("")
            elif max_val is not None and val == max_val:
                styles.append(
                    "background-color: #90EE90; font-weight: bold; color: #006400"
                )  # Light green bg, dark green text
            elif min_val is not None and val == min_val:
                styles.append(
                    "background-color: #FFB6C1; font-weight: bold; color: #8B0000"
                )  # Light red bg, dark red text
            else:
                styles.append("")
        return styles
    else:
        return [""] * len(s)


def weighted_avg(group):
    """
    Calculate weighted average from a group that has both acc_norm and num_samples
    """
    if len(group) == 0:
        return "___"

    # group is a DataFrame with both columns
    acc_values = group["acc_norm"]
    weights = group["num_samples"]

    # Remove NaN values
    valid_mask = ~(pd.isna(acc_values) | pd.isna(weights)) & (weights > 0)

    if valid_mask.sum() > 0:
        return np.average(acc_values[valid_mask], weights=weights[valid_mask])
    else:
        return "___"


def get_styled_df(summaries, columns=["task_pretty_name", "langs"]):
    pivot_df = (
        summaries.groupby(["model_name", *columns])
        .apply(weighted_avg)
        .reset_index()  # Convert back to DataFrame
        .rename(columns={0: "weighted_avg"})  # Name the result column
        .pivot_table(index="model_name", columns=columns, values="weighted_avg")
    )

    # Add aggregate columns
    pivot_df["Mean"] = pivot_df.mean(axis=1)
    # Sort by mean performance
    pivot_df = pivot_df.sort_values("Mean", ascending=False)

    # add column-wise stats
    pivot_df.loc["─" * 20] = np.nan  # Separator row
    pivot_df.loc["MEAN"] = pivot_df.mean(axis=0)
    pivot_df.loc["STD"] = pivot_df.std(axis=0)

    styled_df = (
        pivot_df.style.apply(highlight_extremes, axis=0)
        .format(precision=3)
        .set_table_styles(
            [
                {
                    "selector": "th.level0",
                    "props": [
                        ("border-right", "3px solid black"),
                        ("text-align", "center"),
                    ],
                },
                {
                    "selector": "td",
                    "props": [("text-align", "center"), ("font-size", "11px")],
                },
            ]
        )
    )
    styled_df = styled_df.format(na_rep="─────")
    return styled_df

## Structural Text Elements

In [417]:
styling_categories = [
    "Scripted text",
    "Macron Diacritic",
    "Fullwidth Characters",
    "Upside Down/Rotated",
    "Double struck",
    "Enclosed Characters",
    "Strikethrough",
    "Diacriticized styling",
]
style_summaries = all_summaries[
    (all_summaries["filtering_mode"] == FILTER)
    & all_summaries["task_pretty_name"].isin(styling_categories)
]
style_summaries

,filtering_mode,model_name,task,task_pretty_name,canonical_count,num_samples,category,langs,acc_norm,acc_norm_std,acc,acc_std,canonical_task_name,canonical_acc,canonical_acc_norm,group_name
1721,only_canonical_correct,Aya,tokenizer_robustness_completion_english_macron...,Macron Diacritic,133,40,Structural Text Elements,eng_Latn,0.250000,0.438529,0.225000,0.422902,tokenizer_robustness_completion_english_canonical,0.924812,1.0,tokenizer_robustness_completion_english
1723,only_canonical_correct,Aya,tokenizer_robustness_completion_english_script...,Scripted text,133,40,Structural Text Elements,eng_Latn,0.325000,0.474342,0.200000,0.405096,tokenizer_robustness_completion_english_canonical,0.924812,1.0,tokenizer_robustness_completion_english
1758,only_canonical_correct,Aya,tokenizer_robustness_completion_stem_diacritic...,Diacriticized styling,11,47,Structural Text Elements,eng_Latn,0.425532,0.499769,0.404255,0.496053,tokenizer_robustness_completion_stem_canonical,1.000000,1.0,tokenizer_robustness_completion_stem
1760,only_canonical_correct,Aya,tokenizer_robustness_completion_stem_double_st...,Double struck,15,15,Structural Text Elements,eng_Latn,0.400000,0.507093,0.200000,0.414039,tokenizer_robustness_completion_stem_canonical,0.933333,1.0,tokenizer_robustness_completion_stem
1761,only_canonical_correct,Aya,tokenizer_robustness_completion_stem_enclosed_...,Enclosed Characters,11,36,Structural Text Elements,eng_Latn,0.305556,0.467177,0.166667,0.377964,tokenizer_robustness_completion_stem_canonical,1.000000,1.0,tokenizer_robustness_completion_stem
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3295,only_canonical_correct,mBERT,tokenizer_robustness_completion_stem_enclosed_...,Enclosed Characters,11,36,Structural Text Elements,eng_Latn,0.194444,0.401386,0.083333,0.280306,tokenizer_robustness_completion_stem_canonical,1.000000,1.0,tokenizer_robustness_completion_stem
3296,only_canonical_correct,mBERT,tokenizer_robustness_completion_stem_fullwidth...,Fullwidth Characters,13,13,Structural Text Elements,eng_Latn,0.307692,0.480384,0.384615,0.506370,tokenizer_robustness_completion_stem_canonical,0.923077,1.0,tokenizer_robustness_completion_stem
3297,only_canonical_correct,mBERT,tokenizer_robustness_completion_stem_scripted_...,Scripted text,14,14,Structural Text Elements,eng_Latn,0.285714,0.468807,0.142857,0.363137,tokenizer_robustness_completion_stem_canonical,0.928571,1.0,tokenizer_robustness_completion_stem
3298,only_canonical_correct,mBERT,tokenizer_robustness_completion_stem_strikethr...,Strikethrough,11,44,Structural Text Elements,eng_Latn,0.295455,0.461522,0.181818,0.390154,tokenizer_robustness_completion_stem_canonical,1.000000,1.0,tokenizer_robustness_completion_stem


In [418]:
save_path = OUTPUT_DIR / "style_summary.tsv"

style_summaries.to_csv(save_path, sep="\t")
print(f"File saved at \n{save_path}")

File saved at 
output/results-v5/summary/all/style_summary.tsv


In [419]:
styled_df = get_styled_df(style_summaries, columns=["task_pretty_name", "langs"])

# Set caption
styled_df = styled_df.set_caption(
    f"Unicode Characters (Green=Best, Red=Worst) {FILTER}"
)
styled_df

/var/folders/8n/l2sy_xnn4fv5j3b8z0dhfxlh0000gn/T/ipykernel_35969/2339757082.py:97: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(weighted_avg)


task_pretty_name,Diacriticized styling,Double struck,Enclosed Characters,Fullwidth Characters,Macron Diacritic,Scripted text,Strikethrough,Upside Down/Rotated,Mean
langs,eng_Latn,eng_Latn,eng_Latn,eng_Latn,eng_Latn,eng_Latn,eng_Latn,eng_Latn,
model_name,,,,,,,,,
XGLM,0.387755,0.933333,0.777778,1.000000,0.275000,0.963636,0.295455,0.142857,0.596977
GPT-2,0.489796,0.333333,0.361111,0.428571,0.435897,0.407407,0.386364,0.500000,0.417810
Phi-3,0.489796,0.200000,0.277778,0.500000,0.300000,0.400000,0.477273,0.571429,0.402034
Aya,0.425532,0.400000,0.305556,0.357143,0.250000,0.290909,0.409091,0.642857,0.385136
GPT-4o,0.442308,0.294118,0.333333,0.375000,0.375000,0.368421,0.479167,0.375000,0.380293
Comma,0.468085,0.200000,0.277778,0.428571,0.358974,0.314815,0.522727,0.428571,0.374940
Llama-3.2,0.428571,0.312500,0.250000,0.400000,0.350000,0.303571,0.477273,0.466667,0.373573
ByT5,0.469388,0.400000,0.277778,0.500000,0.256410,0.388889,0.409091,0.285714,0.373409


## Noise

In [420]:
multilingual_noise = [
    "Keyboard proximity errors",
    "Space removal",
    "Character deletion",
    "Typographical errors",
    "OCR Errors",
]

FILTER = "only_canonical_correct"
multilingual_noise_summaries = all_summaries[
    (all_summaries["filtering_mode"] == FILTER)
    & (all_summaries["task_pretty_name"].isin(multilingual_noise))
    # & (~all_summaries["task"].str.contains("stem"))
]

styled_df = get_styled_df(multilingual_noise_summaries)
styled_df

/var/folders/8n/l2sy_xnn4fv5j3b8z0dhfxlh0000gn/T/ipykernel_35969/2339757082.py:97: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(weighted_avg)


## Math

In [ ]:
math_perturbs = [
    "Spelled out",
    "Superscript/subscript",
    "LaTeX",
    "Decorative Unicode",
    "Farsi",
    "Turkish",
    "Italian",
    "Chinese",
]

FILTER = "only_canonical_correct"
math_summaries = all_summaries[
    (all_summaries["filtering_mode"] == FILTER)
    & (all_summaries["task_pretty_name"].isin(math_perturbs))
    & (all_summaries["task"].str.contains("math"))
]

styled_df = get_styled_df(math_summaries, ["task_pretty_name"])
styled_df = styled_df.set_caption(f"MATH {FILTER}")
styled_df

/var/folders/8n/l2sy_xnn4fv5j3b8z0dhfxlh0000gn/T/ipykernel_35969/2339757082.py:97: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(weighted_avg)


task_pretty_name,Chinese,Decorative Unicode,Farsi,Italian,LaTeX,Spelled out,Turkish,Mean
model_name,,,,,,,,
Phi-3,0.764706,0.470588,0.705882,1.000000,0.882353,0.294118,0.941176,0.722689
mBERT,0.857143,0.428571,0.642857,0.785714,0.928571,0.357143,0.714286,0.673469
ByT5,0.733333,0.533333,0.866667,0.600000,0.866667,0.333333,0.666667,0.657143
GPT-4o,0.842105,0.421053,0.578947,0.894737,0.789474,0.368421,0.684211,0.654135
Tekken,0.647059,0.529412,0.764706,0.882353,0.647059,0.352941,0.705882,0.647059
Aya,0.642857,0.357143,0.714286,0.714286,0.785714,0.428571,0.857143,0.642857
TokenMonster,0.785714,0.571429,0.500000,0.714286,0.785714,0.428571,0.714286,0.642857
Llama-3.2,0.764706,0.352941,0.647059,0.882353,0.882353,0.352941,0.470588,0.621849
XGLM,0.733333,0.733333,0.533333,0.666667,0.733333,0.333333,0.600000,0.619048


In [433]:
latex_perturbs = [
    # "Spelled out",
    # "Superscript/subscript",
    "LaTeX",
    # "Decorative Unicode",
    # "Farsi",
    # "Turkish",
    # "Italian",
    # "Chinese",
]

FILTER = "only_canonical_correct"
math_summaries = all_summaries[
    (all_summaries["filtering_mode"] == FILTER)
    & (all_summaries["task_pretty_name"].isin(latex_perturbs))
    # & (all_summaries["task"].str.contains("math"))
]

styled_df = get_styled_df(math_summaries, ["task_pretty_name", "group_name"])
styled_df = styled_df.set_caption(f"LaTeX {FILTER}")
styled_df

/var/folders/8n/l2sy_xnn4fv5j3b8z0dhfxlh0000gn/T/ipykernel_35969/2339757082.py:97: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(weighted_avg)


## Morphological

In [426]:
morph_perturbs = [
    "Contractions",
        "Inflections",
        "Compounds",
        "Derivations",
        "Morpheme separation",
        # "Spelled out",
]

FILTER = "only_canonical_correct"
morph_summaries = all_summaries[
    (all_summaries["filtering_mode"] == FILTER)
    & (all_summaries["task_pretty_name"].isin(morph_perturbs))
    # & (all_summaries["task"].str.contains("math"))
]

styled_df = get_styled_df(morph_summaries, ["task_pretty_name", "langs"])
styled_df = styled_df.set_caption(f"Morphological Challenges, {FILTER}")
styled_df

/var/folders/8n/l2sy_xnn4fv5j3b8z0dhfxlh0000gn/T/ipykernel_35969/2339757082.py:97: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(weighted_avg)
